# Experiment 4.0.4 — Hidden-state memory vs fully-spiking readout

This notebook is analysis-only. The 20 probes are produced by the Slurm array runner.

Question: for the frozen Exp4.0.1 RSNN H128 / tau=250 ms checkpoints, especially Multi-H `(31,1)`, does poor fully-spiking WholeCount accuracy reflect poor recurrent hidden memory or information loss after the hidden endpoint state?

The Linear probe sees only one 128-D post-reset hidden membrane vector at the causal valid endpoint. It does not see phase-wise Fixed250 vectors or output-neuron states.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'AGENTS.md').exists():
    repo_root = repo_root.parent
root = repo_root / 'notebooks' / 'artifacts' / 'experiment_4_0_4_hidden_state_probe' / 'rsnn_cap_variant_endpoint_state_probe_v1'
summary = pd.read_csv(root / 'summary.csv')
runs = pd.read_csv(root / 'runs.csv')
paired = pd.read_csv(root / 'paired_effects.csv')
summary

## Primary diagnostic

Compare each variant's original fully-spiking valid WholeCount BA with the post-hoc endpoint hidden-state Linear-probe BA. A large positive `probe_minus_fully_spiking_ba` means class information is available in the recurrent hidden endpoint state but is not fully exposed by the existing hidden-to-output spiking/readout path.

In [ ]:
test = runs[runs['split'] == 'test'].copy()
test['probe_minus_fully_spiking_ba'] = test['state_probe_balanced_accuracy'] - test['valid_count_balanced_accuracy']
display_cols = [
    'variant', 'hidden_cap', 'output_cap', 'seed',
    'valid_count_balanced_accuracy',
    'state_probe_balanced_accuracy',
    'probe_minus_fully_spiking_ba',
]
test[display_cols].sort_values(['variant', 'seed'])

In [ ]:
plot_data = test.groupby('variant')[['valid_count_balanced_accuracy', 'state_probe_balanced_accuracy']].mean()
ax = plot_data.plot(kind='bar', figsize=(9, 5))
ax.set_ylabel('Mean test balanced accuracy')
ax.set_xlabel('Exp4.0.1 cap variant')
ax.set_title('Fully-spiking WholeCount vs endpoint hidden-state Linear probe')
ax.legend(['Fully-spiking WholeCount', 'Endpoint hidden state + Linear'])
plt.tight_layout()
plt.show()

## Multi-H interpretation

Focus on `multi_h` `(hidden cap=31, output cap=1)`.

- Probe BA close to its poor fully-spiking BA: the recurrent hidden state itself is poor.
- Probe BA much higher than fully-spiking BA: useful memory exists before the binary output bottleneck.
- Probe BA near Multi-HO probe BA while fully-spiking Multi-H remains much worse: strong evidence that the output cap/readout path, rather than hidden memory, explains most of the Multi-H failure.

In [ ]:
paired.sort_values(['variant', 'seed'])